# Chest X-ray to report by a DAG tensor: a CNN and a tiny LM stitched

*ai · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/ai/cxr_report_dag)

A teaching toy, not clinical report generation. A small CNN learns six findings from 410 NIH chest X-rays, and a tiny character-level GPT learns template reports ("enlarged heart. pleural effusion. no pneumothorax.") from a written label header, each on its own. One DAG edge joins them: a connector turns the CNN's finding probabilities into a soft header, so the LM writes a report from the image alone. The training process is itself a tensor DAG (rules on levels, max-plus critical path, one batched call per rule over the seeds), compared with the same network trained end to end with the same compute, then fine-tuned on 251 real texts (figure caption + the sentences citing that radiograph) from openly licensed (CC BY) PubMed Central articles. Stage 2 is weak, as expected with so few pairs, and the page says so.

**Vector form**

$$X^0 = G_{\mathrm{img}} \odot C\big(\sigma(\mathrm{CNN}(\mathrm{image}))\big) + G_{\mathrm{txt}} \odot \mathrm{Tok}[\mathrm{text}],\qquad z = \mathrm{LM}(X^0 + P),\qquad \hat y = \mathrm{parse}\big(\mathrm{greedy}(z)\big),\qquad t = d \oplus (A_{\mathrm{rule}} \otimes t)$$

**Index (tensor) form**

$$q_{mnf} = \sigma\big(\mathrm{CNN}_m(x_{mn})\big)_f \qquad c_{mntc} = \mathrm{GELU}(q_{mnf}W^1_{mfh} + S_{mth})\,W^2_{mhc} + b^2_{mc} \qquad X^0_{mntc} = G_{mntx}\,Y^x_{mntc} \qquad \mathcal L_m = -\frac{1}{|\mu_m|}\sum_{n,t}\mu_{mnt}\log \mathrm{softmax}_v z_{mntv}[y_{mnt}] \qquad F_1 = \frac{2\,\mathrm{tp}}{2\,\mathrm{tp} + \mathrm{fp} + \mathrm{fn}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $A_{\mathrm{rule}}$ | `rule_graph(parse_snakefile(training.smk))` | (R, R) | rule $s$ reads what rule $r$ writes; the training process parsed from data/training.smk with the Circle of Willis problem's parser |
| $\ell_r$ | `dag.levels(A_rule)` | (R,) | level of each rule: CNN, LM and monolith on 0, connector on 1, fine-tune and router on 2, captions on 3, evaluate on 4 |
| $t = d \oplus (A \otimes t)$ | `dag.finish_times(A_rule, seconds)` | (R,) | max-plus finish times from the measured rule durations; $\max t$ is the critical path |
| $q_{mnf}$ | `cnn_output(x, w)` | (M, N, 6) | the CNN's six finding probabilities (sigmoid, multi-label) for each image; the model axis $m$ becomes the groups of one grouped convolution per layer |
| $c_{mntc}$ | `connector(q, slots, w)` | (M, P, d) | the DAG edge: finding probabilities plus a slot vector $S_t$ → an LM input vector; the $K = 6$ soft tokens replace the written label header |
| $G_{mntx}$ | `route_fixed(...), route_learned(r, w)` | (M, N, T, 2) | router: one-hot by data type (header slot or letter), or a learned softmax over the position features (trained dense, used top-1) |
| $X^0 = G_{mntx}Y^x_{mntc}$ | `einsum('mntx,xmntc->mntc', G, Y)` | (M, N, T, d) | repeated $x$ ⇒ summed: the combine of the MoE layer; each expert $Y^x$ is computed only on its dispatched group |
| $z_{mntv}$ | `lm_logits(X0, w)` | (M, N, T, V) | next-letter logits of the tiny GPT (einsum attention with a model axis, from chain_of_thought) |
| $\mathcal L_m$ | `masked_ce(z, y, mask)` | (M,) | per-model loss on the report letters after the header (nats); summing the $M$ losses gives every model its own gradient |
| $\mathrm{tp}_{mf}$ | `einsum('mnf,mnf->mf', pred, true)` | (M, 6) | repeated $n$ ⇒ summed: true positives of the labels parsed back from each written report; micro $F_1$ pools them over $f$ |

What this is. A teaching toy, not clinical report generation: tiny models, 64×64 images, a few hundred noisy labels, and no real radiology reports. Nothing here says anything about clinical accuracy. The idea (networks trained separately, connected through a DAG tensor, optimised globally, the whole process itself controlled by the DAG tensor) is Jacky Ka Long Ko's; this problem carries the CNN + tiny-LM stitching of the previous problem in this series over to chest radiographs and a text that reads like a (very short) report. Why no real reports. None of the openly available report collections passes the licence rule used here (open, redistributable, commercial use allowed): IU X-ray / OpenI (Demner-Fushman et al. 2016) is CC BY-NC-ND; MIMIC-CXR (Johnson et al. 2019), VinDr-CXR and BRAX need credentialed access under data use agreements; CheXpert and PadChest are for research use only. So stage 1 uses template reports generated in code from the NIH label sets, which turns it into multi-label classification verbalised as text, and stage 2 uses figure captions plus the sentences that cite the figure from CC BY PubMed Central articles: short, heterogeneous texts written for case reports, often pointing at arrows and panels. What is new and what is not. Report generation from chest X-rays is an active field (Jing et al. 2018, R2Gen by Chen et al. 2020); a pretrained image encoder joined to a language model by a small learned connector is how LLaVA (Liu et al. 2023) and BLIP-2 (Li et al. 2023) are built. What this problem adds is the orchestration: the model graph and the training process are both run by the tensor DAG engine of the Circle of Willis workflow problem (rule graph parsed from data/training.smk, Kronecker job graph, boolean levels, max-plus critical path), every rule one batched call over the seeds, and the DAG-built model is measured against a monolith with the same compute. Data. NIH ChestX-ray14 (Wang et al. 2017, NIH Clinical Center; no restrictions on use, with attribution): 549 frontal PA images, one per patient, six findings (cardiomegaly, effusion, infiltration, atelectasis, pneumothorax, mass or nodule) or none, at most two per image; 410 training, 139 test. The labels were mined from reports by NLP and are noisy (about 10 % or more wrong). PMC: 316 chest radiograph figures from 316 open-access articles, every one CC BY (checked in the article's own XML licence; rule: CC BY, CC BY-SA or CC0, reject NC, ND, other or missing). Candidates came from the ROCOv2 metadata (Rückert et al. 2024), used only as an index of PMCIDs; nothing from ROCOv2 is bundled. Of 2114 articles checked, 1575 were accepted and 539 rejected by licence (all CC BY-NC). The stage-2 text is the caption followed by the same article's sentences that cite the figure (315 of 316 articles have at least one), whole sentences only, up to 360 characters, the context of the tiny LM: median 35 words (IQR 26–45) against 12 (8–18) for the caption alone. Every figure and text is credited in data/ATTRIBUTION.csv; data/SOURCES.md has the sources, terms, counts, checksums and processing. Models and the training DAG. CNN: three grouped convolutions, 41 014 weights, six sigmoid outputs (multi-label, positives weighted by their rarity). LM: a two-layer character GPT (d = 32, four heads, 40 108 weights) trained on every template, the label header written as six letters in front ("ce----", then "enlarged heart. pleural effusion."). Connector (2848 weights): from the six probabilities $q$ and a slot vector $S_t$ it makes the $K = 6$ soft tokens that replace the header. Levels: 0 train_cnn ∥ train_lm ∥ train_monolith; 1 train_connector (CNN and LM frozen); 2 global_finetune ∥ train_router; 3 caption_finetune (connector + LM on the PMC texts, CNN frozen); 4 evaluate. The tests check the levels and the critical path, that the job graph is nilpotent, that two seeds in one stack train exactly as each alone (to float32 rounding), that one model's loss sends no gradient to the others, that the connector stage changes only the connector, and that the MoE layer with the fixed router equals the DAG wiring to 1e-6, gradients too. Reports are written greedily (with a key/value cache, tested against the plain forward pass) and parsed back into labels by keyword, with negation ("no pneumothorax."). Stage 1 results (2 seeds, 139 test images, seed means; results/stages.csv has every number). The CNN alone: mean AUROC 0.62 (chance 0.5), micro F1 0.31 at threshold 0.5 (random guessing at the training prevalence: 0.16). Its labels written as a header for the LM alone: micro F1 0.32, so the LM verbalises faithfully (0.90 of its outputs are well-formed templates, even for label sets it never saw). The stitched model: micro F1 0.22 after the connector stage and 0.20 after the global fine-tune (0.15 to 0.25 over runs and seeds), every report a well-formed template, 0.26 to 0.27 exactly the template of the true labels. The monolith, the same network trained end to end on the 410 image–report pairs with the same multiply-adds (405 joint steps), writes "clear lungs." or "normal chest." for every image: micro F1 0, exact 0.27 (the share of normal images). So the DAG pipeline clearly beats the monolith, because it reuses the CNN's label supervision and all the template text, but it does not beat the CNN alone: a template report can only re-express what the classifier knows, and the stitched model, trained by likelihood, writes the most probable report, which leans to "normal" when the image is ambiguous, while the CNN's rarity-weighted threshold favours recall, which $F_1$ rewards. The fine-tune changes little. DAG edge = MoE dispatch. The input layer as a mixture of experts (image expert = CNN + connector, text expert = token embedding) with $G$ one-hot by data type gives the DAG-wired outputs exactly (tested). A linear router trained as a dense mixture with the experts frozen, then used top-1 (route, then run the chosen expert), sends every position to its modality's expert (routing accuracy 1.00 in both seeds), so its reports and scores equal the fixed router's. Stage 2 results (65 held-out PMC figures): weak, as expected. After 150 steps on 251 texts the model writes "chest x-ray showing" and then letter soup: BLEU-1 0.05, ROUGE-L 0.06, finding-keyword recall 0, held-out cross-entropy 2.32 nats per letter. Giving it another figure's image changes nothing (2.32 nats, the same scores): the image is not used. A blind baseline that returns one fixed training text scores higher (BLEU-1 0.17, ROUGE-L 0.22). A character LM of 40 000 weights cannot learn this language from about 70 000 characters in a CPU minute; with the first-clause captions of an earlier version (median 6 words) ROUGE-L was about 0.28, but those texts were barely sentences. The DAG machinery works; the data are far too few for stage 2. Time. On a 6-core desktop CPU training takes about 85 s when the machine is quiet, plus about 30 s for the GIF; measured times grow two to three times when other heavy jobs share the machine (the run shown in the GIF and timings.csv was made under such load, so its seconds are larger), with the caption stage the most affected. With one worker per rule the max-plus critical path is CNN → connector → fine-tune → captions → evaluate. The code picks a GPU when torch sees one (pixi run -e cuda); the default environment has a CPU build of torch, so every bundled number is from the CPU. Try. More caption steps (stage 2 improves slowly), no global fine-tune (0 steps), fewer CNN steps (the CNN caps everything), or one seed to see how much the seeds disagree.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/ai/cxr_report_dag/main.py
"""Main loop: parse the training rule graph → run it with the tensor DAG engine (each rule one batched call over the
seeds) → label F1 of the written reports for every stage vs the CNN alone and the monolith, the learned router,
caption metrics after the PMC stage, the measured schedule → plot."""

import time
from pathlib import Path

import numpy as np
import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation as eq
from . import plot, solve
from . import tensors as T

LAST: dict = {}  # the results of the latest run (read by the full-run test)


def _pm(x: torch.Tensor) -> str:
    """'mean ± half-range' over the seeds."""
    return f"{float(x.mean()):.3f} ± {float(x.max() - x.min()) / 2:.3f}"


def _names(y) -> str:
    return ", ".join(T.FINDINGS[f] for f in range(6) if y[f]) or "no finding"


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    f = 0.03 if fast else 1.0
    cfg = solve.Config(
        seeds=list(range(1 if fast else int(params["seeds"]))),
        steps_cnn=max(2, int(params["steps_cnn"] * f)),
        steps_lm=max(2, int(params["steps_lm"] * f)),
        steps_con=max(2, int(params["steps_con"] * f)),
        steps_ft=max(2, int(params["steps_ft"] * f)),
        steps_router=max(2, int(params["steps_router"] * f)),
        steps_cap=max(2, int(params["steps_cap"] * f)),
        lr_ft=params["lr_ft"],
        lr_cap=params["lr_cap"],
        n_test=16 if fast else None,
    )
    S = len(cfg.seeds)
    d = solve.build_dag(cfg)
    lev = dict(zip(d["names"], d["levels"].tolist()))
    print(f"training DAG ({T.TRAINING_GRAPH.name}): {len(d['names'])} rules, {len(d['A'])} jobs, "
          f"{int(d['A'].sum())} job edges; A nilpotent: {solve.dag.is_nilpotent(d['A'])}")  # fmt: skip
    for k in range(int(d["levels"].max()) + 1):
        print(f"  level {k}: " + " ∥ ".join(n for n, lv in lev.items() if lv == k))
    t0 = time.perf_counter()
    st, seconds, d = solve.run_engine(cfg, dev, d)
    total = time.perf_counter() - t0
    sch = solve.schedule(d, seconds)
    data, pmc = st["data"], st["pmc"]
    print(f"device {dev}; {S} seeds in every batched call; NIH {len(data['y_train'])} train / "
          f"{len(data['y_test'])} test images, PMC {len(pmc['cap_train'])} train / {len(pmc['cap_test'])} test "
          "captions")  # fmt: skip
    for n in d["names"]:
        print(f"  {n:16s} level {lev[n]}  {seconds[n]:6.1f} s")
    print(f"serial (one worker, level by level) {sch['serial']:.1f} s; with one worker per rule the max-plus "
          f"critical path is {' → '.join(sch['path'])} = {sch['makespan']:.1f} s")  # fmt: skip
    mono_count = sum(T.count(st["train_monolith"][k]) for k in ("cnn", "con", "lm"))
    print(f"weights per model: CNN {T.count(st['train_cnn']['w'])}, connector {T.count(st['train_connector']['con'])}"
          f", LM {T.count(st['train_lm']['w'])}; monolith {mono_count} (the same architecture)")  # fmt: skip
    print(f"budget: the DAG pipeline takes {cfg.steps_cnn} CNN + {cfg.steps_lm} LM + {cfg.steps_con} connector + "
          f"{cfg.steps_ft} fine-tune steps of {cfg.batch}; the monolith gets the same multiply-adds: {cfg.steps_mono}"
          " joint steps")  # fmt: skip

    ev = st["evaluate"]
    f1_micro = {"cnn": ev["cnn_micro"], **{k: r["micro"] for k, r in ev["rep"].items()}}
    f1_macro = {"cnn": ev["cnn_macro"], **{k: r["macro"] for k, r in ev["rep"].items()}}
    print(f"CNN alone: mean AUROC over the six findings {_pm(ev['auc'].mean(1))} (chance 0.5)")
    print(f"NIH test, labels parsed back from the written report (micro F1; random guessing at the training "
          f"prevalence {ev['chance']:.3f}), mean ± half-range over seeds:")  # fmt: skip
    print(f"  {'stage':16s} {'micro F1':>16s} {'macro F1':>16s} {'well-formed':>16s} {'exact template':>16s}")
    for k in plot.STAGE:
        r = ev["rep"].get(k)
        extra = f"{_pm(r['wellformed']):>16s} {_pm(r['exact']):>16s}" if r else f"{'—':>16s} {'—':>16s}"
        print(f"  {k:16s} {_pm(f1_micro[k]):>16s} {_pm(f1_macro[k]):>16s} {extra}")
    print(f"learned router (trained dense, used top-1): share of positions sent to the modality's expert "
          f"{_pm(ev['route_acc'])}")  # fmt: skip
    for key, label in (("caponly", "caption alone"), ("cap", "caption + citing sentences (the stage-2 target)")):
        w = np.array([len(c.split()) for c in pmc[f"{key}_train"] + pmc[f"{key}_test"]])
        print(f"PMC text, {label}: words median {np.median(w):.0f}, IQR {np.percentile(w, 25):.0f}–"
              f"{np.percentile(w, 75):.0f}, range {w.min()}–{w.max()}")  # fmt: skip
    print(f"  ({pmc['cut']} texts whose first sentence alone exceeds {T.MAX_CAP} characters were cut at a word)")
    print("PMC test texts (stage 2):")
    print(f"  {'model':18s} {'BLEU-1':>14s} {'ROUGE-L':>14s} {'kw recall':>14s} {'kw precision':>14s} "
          f"{'nats/letter':>14s}")  # fmt: skip
    for k, c in ev["cap"].items():
        print(f"  {k:18s} " + " ".join(f"{_pm(c[m]):>14s}" for m in ("bleu1", "rouge_l", "kw_recall",
                                                                     "kw_precision", "nats")))  # fmt: skip
    print(f"  (constant = the training caption closest to all others: '{ev['constant']}')")

    # examples: the first NIH test image of four label groups (whatever the model writes), the first PMC test figures
    with np.load(T.DATA / "nih_cxr_subset.npz") as z:
        raw = z["images"][z["split"] == 1]
    with np.load(T.DATA / "pmc_cxr_subset.npz") as z:
        raw_p = z["images"][z["split"] == 1]
    y_test = data["y_test"][: len(ev["txt"]["finetune"][0])].int().tolist()
    examples = []
    for want in ((0,) * 6, (1, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0), (0, 0, 0, 0, 0, 1)):
        hits = [i for i, y in enumerate(y_test) if tuple(y) == want] or [len(examples)]
        i = hits[0]
        dag_txt = ev["txt"]["finetune"][0][i]
        examples.append({"img": raw[i], "q": ev["q"][0, i].numpy(), "dag": dag_txt or "(nothing)",
                         "mono": ev["txt"]["monolith"][0][i] or "(nothing)", "pred": T.parse_report(dag_txt),
                         "truth": tuple(y_test[i]), "truth_short": _names(y_test[i])})  # fmt: skip
        print(f"  NIH test image {i} ({_names(y_test[i])}): DAG '{dag_txt}' | monolith "
              f"'{ev['txt']['monolith'][0][i]}'")  # fmt: skip
    ct = ev["cap_txt"]
    cap_examples = [{"img": raw_p[i], "truth": pmc["cap_test"][i], "gen": ct["caption"][0][i] or "(nothing)"}
                    for i in range(min(2, len(ct["caption"][0])))]  # fmt: skip
    for i, ex in enumerate(cap_examples):
        print(f"  PMC test figure {i}: true '{ex['truth']}' | model '{ex['gen']}'")
    print(f"total {total:.0f} s")

    tok = eq.position_items(st["table"]["tokens"][:12, :-1][None])[1]
    G_fixed = eq.route_fixed(1, 12, tok)[0, :, :, 0].cpu().numpy()
    res = {"names": d["names"], "levels": d["levels"].tolist(), "A_rule": d["A_rule"].tolist(), "seconds": seconds,
           "path": sch["path"], "finish": sch["finish"].tolist(), "makespan": sch["makespan"], "serial": sch["serial"],
           "loss": {n: st[n]["loss"] for n in d["names"] if n != "evaluate"}, "seeds": cfg.seeds,
           "f1_micro": f1_micro, "f1_macro": f1_macro, "wellformed": {k: r["wellformed"] for k, r in ev["rep"].items()},
           "exact": {k: r["exact"] for k, r in ev["rep"].items()}, "auc": ev["auc"], "route_acc": ev["route_acc"],
           "route_show": float(ev["route_acc"][0]), "chance": ev["chance"], "cap": ev["cap"], "G_fixed": G_fixed,
           "G_learned": ev["route_p"][0, :12].numpy(), "examples": examples, "cap_examples": cap_examples,
           "cap_rows": [(pmc["cap_test"][i], ct["finetune"][0][i], ct["caption"][0][i], ct["caption_shuffled"][0][i])
                        for i in range(len(ct["caption"][0]))],
           "rep_rows": [(_names(y_test[i]), " ".join(f"{v:.2f}" for v in ev["q"][0, i].tolist()),
                         ev["txt"]["finetune"][0][i], ev["txt"]["monolith"][0][i])
                        for i in range(len(y_test))]}  # fmt: skip
    LAST.clear()
    LAST.update(res)
    plot.write_tables(res, out_dir)
    plot.schedule_png(res, out_dir / "schedule.png")
    return plot.animate(res, out_dir / "cxr_report_dag.gif", n_frames=3 if fast else 14, hold=2 if fast else 18)


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/ai/cxr_report_dag/equation.py
r"""A chest X-ray CNN and a tiny report LM stitched by one DAG edge, the same edge as a mixture-of-experts layer, and
the metrics (label F1 of generated reports, AUROC, BLEU-1 and ROUGE-L of captions).

Indices: m = model on the stacked axis (seed), n = example, t = input position, c = LM channel (d), f = finding
(six), h = hidden channel, v = vocabulary letter, x = expert (img, txt).

    q_{mnf} = \sigma(\mathrm{CNN}_m(\mathrm{image}_{mn}))_f                      (M, N, 6) finding probabilities
    c_{mntc} = \mathrm{GELU}(q_{mnf} W^1_{mfh} + S_{mth}) W^2_{mhc} + b^2_{mc}   (f, h summed) connector, slot t < K
    X^0_{mntc} = c_{mntc} for t < K,   \mathrm{Tok}_{m, t_{mnt}, c} for t ≥ K       the DAG edge: a soft label header
    z_{mntv} = \mathrm{LM}_m(X^0 + P)                                          next-letter logits

As an MoE layer at the LM input: X^0_{mntc} = G_{mntx} Y^x_{mntc} (x summed) with the fixed router
G_{mnt,img} = [t < K], or a learned router p = softmax(r W^R + b^R) trained dense and used top-1.

The LM blocks come from problems/ai/chain_of_thought; the token lookup, connector, LM read-out, routers, dispatch and
masked cross-entropy are imported from problems/ai/stitch_llm_cnn_dag (the same functions, this problem's shapes).
"""

import torch
import torch.nn.functional as F

from problems.ai.chain_of_thought.equation import layer_norm, mlp
from problems.ai.stitch_llm_cnn_dag.equation import (  # noqa: F401
    balance_loss,
    connector,
    dispatch,
    embed,
    lm_logits,
    masked_ce,
    route_fixed,
    route_learned,
)

from . import tensors as T


# ---------------------------------------------------------------- the CNN (one grouped convolution per layer)
def cnn_features(x: torch.Tensor, w: dict) -> torch.Tensor:
    """Images x (M, N, 64, 64) → features (M, N, E); the model axis becomes conv groups, so models never mix."""
    m, n = x.shape[:2]
    z = x.transpose(0, 1)  # (N, M, 64, 64): one input channel per model
    z = F.max_pool2d(F.relu(F.conv2d(z, w["c1"].flatten(0, 1), w["b1"].flatten(), groups=m)), 2)  # (N, 8M, 30, 30)
    z = F.max_pool2d(F.relu(F.conv2d(z, w["c2"].flatten(0, 1), w["b2"].flatten(), groups=m)), 2)  # (N, 16M, 14, 14)
    z = F.max_pool2d(F.relu(F.conv2d(z, w["c3"].flatten(0, 1), w["b3"].flatten(), groups=m)), 2)  # (N, 16M, 6, 6)
    z = z.reshape(n, m, 576).transpose(0, 1)  # (M, N, 576)
    return F.relu(torch.einsum("mnk,mke->mne", z, w["fc"]) + w["bf"][:, None])


def cnn_logits(h: torch.Tensor, w: dict) -> torch.Tensor:
    """Finding logits (M, N, 6) from the features h (M, N, E)."""
    return torch.einsum("mne,mef->mnf", h, w["head"]) + w["bh"][:, None]


def cnn_output(x: torch.Tensor, w: dict) -> torch.Tensor:
    """q = σ(CNN logits): six independent finding probabilities, what the CNN passes along the DAG edge."""
    return torch.sigmoid(cnn_logits(cnn_features(x, w), w))


def bce(logits: torch.Tensor, y: torch.Tensor, pos_weight: torch.Tensor) -> torch.Tensor:
    """Multi-label loss per model (M,): binary cross-entropy, positives weighted by (negatives / positives)."""
    return F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight, reduction="none").mean((1, 2))


# ---------------------------------------------------------------- the DAG-wired model
def soft_prefix(x: torch.Tensor, w_cnn: dict, w_con: dict) -> torch.Tensor:
    """CNN → connector: the K soft tokens of each image.  x (M, N, 64, 64) → (M, N, K, d)."""
    m, n = x.shape[:2]
    q = cnn_output(x, w_cnn)  # (M, N, 6)
    qk = q[:, :, None].expand(m, n, T.K, q.shape[-1]).reshape(m, n * T.K, -1)
    slots = torch.arange(T.K, device=x.device).repeat(n)[None].expand(m, -1)  # (M, N·K) slot of every row
    return connector(qk, slots, w_con).reshape(m, n, T.K, -1)


def stitched_inputs(x: torch.Tensor, tokens: torch.Tensor, w_cnn: dict, w_con: dict, w_lm: dict) -> torch.Tensor:
    """The DAG edge CNN → connector → LM: K soft tokens from the image replace the label header.
    x (M, N, 64, 64), tokens (M, N, T) → X0 (M, N, T, d)."""
    return torch.cat([soft_prefix(x, w_cnn, w_con), embed(tokens[:, :, T.K :], w_lm)], dim=2)


def stitched_logits(x, tokens, w_cnn, w_con, w_lm) -> torch.Tensor:
    return lm_logits(stitched_inputs(x, tokens, w_cnn, w_con, w_lm), w_lm)


# ---------------------------------------------------------------- the same edge as an MoE layer
def position_items(tokens: torch.Tensor):
    """At each input position an id (the IMG placeholder at the K header slots, else the letter) and whether it
    shows the image.  → ids (M, N, T), is_img (T,) bool."""
    is_img = torch.arange(tokens.shape[2], device=tokens.device) < T.K
    return torch.where(is_img, torch.full_like(tokens, T.STOI[T.IMG]), tokens), is_img


def router_features(x: torch.Tensor, ids: torch.Tensor, is_img: torch.Tensor, blank: torch.Tensor) -> torch.Tensor:
    """r_{mnt} = 8×8 average-pooled image of the position (a blank page at text positions) ⊕ one-hot id."""
    m, n, t = ids.shape
    pooled = F.avg_pool2d(x.reshape(m * n, 1, T.IMG_SIZE, T.IMG_SIZE), 8).reshape(m, n, 1, 64)
    pooled_blank = F.avg_pool2d(blank[None, None], 8).reshape(1, 1, 1, 64)
    img = torch.where(is_img[None, None, :, None], pooled, pooled_blank)
    return torch.cat([img, F.one_hot(ids, len(T.VOCAB)).to(x.dtype)], dim=-1)


def moe_inputs(x, ids, G, w_cnn, w_con, w_lm, blank, q_img=None) -> torch.Tensor:
    """X0_{mntc} = G_{mntx} Y^x_{mntc}: dispatch every position to its expert, run each expert batched on its own
    group (CNN once per image and once on a blank page, connector per routed position; token lookup), scatter back,
    combine. q_img: the CNN outputs if already computed (greedy decoding reuses them at every letter)."""
    m, n, t = ids.shape
    flat_ids = ids.reshape(m, n * t)
    q_img = cnn_output(x, w_cnn) if q_img is None else q_img  # (M, N, 6)
    q_blank = cnn_output(blank.expand(m, 1, T.IMG_SIZE, T.IMG_SIZE), w_cnn)  # (M, 1, 6)
    is_img = torch.arange(t, device=x.device) < T.K
    Y = []
    for x_e in range(2):
        idx, valid = dispatch(G[..., x_e].reshape(m, n * t) > 0)
        if x_e == 0:  # E_img = CNN + connector at slot t of example n (a text position sees a blank page)
            ex, slot = idx // t, idx % t
            q = torch.where(is_img[slot][..., None], q_img.gather(1, ex[..., None].expand(-1, -1, 6)), q_blank)
            out = connector(q, slot.clamp(max=T.K - 1), w_con)
        else:  # E_txt = the LM's own token embedding
            model = torch.arange(m, device=x.device)[:, None]
            out = w_lm["tok"][model, flat_ids.gather(1, idx)]
        out = out * valid[..., None]
        Y.append(torch.zeros(m, n * t, out.shape[-1], dtype=out.dtype, device=x.device).scatter_add(
            1, idx[..., None].expand_as(out), out).reshape(m, n, t, -1))  # fmt: skip
    return torch.einsum("mntx,xmntc->mntc", G, torch.stack(Y))


# ---------------------------------------------------------------- greedy decoding with a key/value cache
def lm_step(X_new: torch.Tensor, w: dict, cache: list, pos: int) -> torch.Tensor:
    r"""One new input vector per sequence through the LM, attending to the cached keys and values of the earlier
    positions (the same numbers lm_logits computes for that position, without recomputing the past).
    X_new (M, N, 1, d), cache[layer] = (K, V) each (M, N, H, pos, d_k) or None → logits (M, N, 1, V).
    S_{ij} = Q_{ik}K_{jk}/\sqrt{d_k} over j ≤ pos (k summed): the causal mask is automatic, the future is not there."""
    X = X_new + w["pos"][:, None, pos : pos + 1]
    for li, p in enumerate(w["layers"]):
        h = layer_norm(X, p["g1"], p["b1"])
        q = torch.einsum("mnic,mhck->mnhik", h, p["Wq"]) + p["bq"][:, None, :, None, :]
        k = torch.einsum("mnic,mhck->mnhik", h, p["Wk"]) + p["bk"][:, None, :, None, :]
        v = torch.einsum("mnic,mhck->mnhik", h, p["Wv"]) + p["bv"][:, None, :, None, :]
        if cache[li] is not None:
            k, v = torch.cat([cache[li][0], k], 3), torch.cat([cache[li][1], v], 3)
        cache[li] = (k, v)
        A = torch.softmax(torch.einsum("mnhik,mnhjk->mnhij", q, k) / q.shape[-1] ** 0.5, -1)
        mix = torch.einsum("mnhij,mnhjk->mnhik", A, v)
        X = X + torch.einsum("mnhik,mhkc->mnic", mix, p["Wo"]) + p["bo"][:, None, None]
        X = X + mlp(layer_norm(X, p["g2"], p["b2"]), p)
    return torch.einsum("mntc,mcv->mntv", layer_norm(X, w["gf"], w["bf"]), w["head"]) + w["bias"][:, None, None]


@torch.no_grad()
def greedy_cached(X_prompt: torch.Tensor, w: dict, n_new: int) -> torch.Tensor:
    """Feed the prompt vectors (M, N, P, d) one by one, then write n_new letters, each the argmax given what is
    written so far; padding after EOS; stop when every row has written EOS.  → ids (M, N, n_new)."""
    cache = [None] * len(w["layers"])
    for t in range(X_prompt.shape[2]):
        z = lm_step(X_prompt[:, :, t : t + 1], w, cache, t)
    m, n = X_prompt.shape[:2]
    out = torch.full((m, n, n_new), T.STOI[T.PAD], dtype=torch.long, device=X_prompt.device)
    done = torch.zeros(m, n, dtype=torch.bool, device=X_prompt.device)
    for k in range(n_new):
        nxt = z[:, :, 0].argmax(-1)
        out[..., k] = torch.where(done, torch.full_like(nxt, T.STOI[T.PAD]), nxt)
        done |= nxt == T.STOI[T.EOS]
        if bool(done.all()) or k == n_new - 1:
            break
        z = lm_step(embed(nxt[..., None], w), w, cache, X_prompt.shape[2] + k)
    return out


# ---------------------------------------------------------------- metrics
def f1(pred: torch.Tensor, true: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Label-level F1 per model: pred (M, N, F) bool, true (N, F) bool → micro F1 (M,), macro F1 (M,).
    tp_{mf} = Σ_n pred·true (n summed); micro pools the counts over f, macro averages the per-finding F1."""
    p, y = pred.float(), true.float()[None]
    tp = torch.einsum("mnf,mnf->mf", p, y.expand_as(p))
    fp, fn = p.sum(1) - tp, y.sum(1) - tp
    per = 2 * tp / (2 * tp + fp + fn).clamp(min=1e-9)
    micro = 2 * tp.sum(1) / (2 * tp.sum(1) + fp.sum(1) + fn.sum(1)).clamp(min=1e-9)
    return micro, per.mean(1)


def auroc(score: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Area under the ROC curve = P(score of a random positive > score of a random negative), ties count ½."""
    pos, neg = score[y > 0], score[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return torch.tensor(float("nan"))
    d = pos[:, None] - neg[None, :]
    return ((d > 0).float() + 0.5 * (d == 0).float()).mean()


def bleu1(hyp: str, ref: str) -> float:
    """Clipped unigram precision × brevity penalty (word level, one reference)."""
    h, r = hyp.split(), ref.split()
    if not h:
        return 0.0
    match = sum(min(h.count(w), r.count(w)) for w in set(h))
    bp = 1.0 if len(h) > len(r) else torch.exp(torch.tensor(1 - len(r) / len(h))).item()
    return bp * match / len(h)


def rouge_l(hyp: str, ref: str) -> float:
    """ROUGE-L F1 = 2·LCS / (|hyp| + |ref|) on words; the LCS by the usual dynamic programme over the two strings."""
    h, r = hyp.split(), ref.split()
    if not h or not r:
        return 0.0
    row = [0] * (len(r) + 1)
    for a in h:
        prev = 0
        for j, b in enumerate(r, 1):
            prev, row[j] = row[j], prev + 1 if a == b else max(row[j], row[j - 1])
    return 2 * row[-1] / (len(h) + len(r))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/ai/cxr_report_dag/tensors.py
"""Build the concrete tensors: the NIH chest X-ray subset, the template reports, the PMC caption pairs, stacked weights.

data/nih_cxr_subset.npz   64×64 NIH ChestX-ray14 frontal (PA) radiographs, one per patient, six findings (multi-hot)
data/pmc_cxr_subset.npz   64×64 chest radiograph figures from CC BY / CC BY-SA / CC0 PubMed Central articles
data/pmc_captions.csv     their captions (from the article XML); data/ATTRIBUTION.csv credits every figure
data/training.smk         the training process as a Snakefile-style rule graph, run by the tensor DAG engine

Every weight tensor has the model axis m first (model m = seed), so one optimiser step trains every seed at once,
each model only touching its own slice. Stacking, minibatch draws and weight-dict helpers are imported from
problems/ai/stitch_llm_cnn_dag.
"""

import csv
import itertools
import math
import re
from pathlib import Path

import numpy as np
import torch

from problems.ai.stitch_llm_cnn_dag.tensors import _stack, clone, count, draws, expand, leaves, select  # noqa: F401

DATA = Path(__file__).parent / "data"
TRAINING_GRAPH = DATA / "training.smk"
IMG_SIZE = 64

FINDINGS = ["cardiomegaly", "effusion", "infiltration", "atelectasis", "pneumothorax", "mass/nodule"]
CODES = "ceiapm"  # one header letter per finding, '-' when absent
NIH_NAMES = {"Cardiomegaly": 0, "Effusion": 1, "Infiltration": 2, "Atelectasis": 3, "Pneumothorax": 4, "Mass": 5,
             "Nodule": 5}  # fmt: skip
PHRASES = [  # two phrasings per finding, in this canonical order
    ["cardiomegaly.", "enlarged heart."],
    ["pleural effusion.", "effusion."],
    ["infiltrate.", "consolidation."],
    ["atelectasis.", "volume loss."],
    ["pneumothorax.", "small pneumothorax."],
    ["nodule.", "mass."],
]
NORMAL = ["normal chest.", "clear lungs."]
NEGATION = "no pneumothorax."
N_VARIANTS = 8  # bit 0: phrasing of sentence 1, bit 1: of sentence 2, bit 2: add "no pneumothorax." when it applies
KEYWORDS = [  # what the parser looks for (also in free text such as captions)
    r"cardiomegaly|enlarged heart|cardiac enlargement|enlarged cardiac",
    r"effusion",
    r"infiltrat|consolidation",
    r"atelecta|volume loss|collapse",
    r"(?<!no )pneumothora",
    r"nodul|mass",
]

K = 6  # header / soft-prefix length: one slot per finding
MAX_TXT = 50  # longest template report (characters)
CTX = K + 1 + MAX_TXT + 1  # header, BOS, text, EOS
MAX_CAP = 360  # longest stage-2 text: caption + the article's sentences citing the figure (characters)
CTX_CAP = K + 1 + MAX_CAP + 1  # the LM's position table covers both
PAD, IMG, BOS, EOS = "_", "#", ">", "|"
VOCAB = PAD + IMG + BOS + EOS + " .,-" + "0123456789" + "abcdefghijklmnopqrstuvwxyz"
STOI = {c: i for i, c in enumerate(VOCAB)}


# ---------------------------------------------------------------- template reports and their parser
def label_sets() -> list[tuple]:
    """Every label set the toy reports: no finding, each single finding, each pair (22 multi-hot tuples)."""
    out = [(0,) * 6]
    for k in (1, 2):
        for c in itertools.combinations(range(6), k):
            out.append(tuple(int(f in c) for f in range(6)))
    return out


SETS = label_sets()
SET_Y = torch.tensor(SETS, dtype=torch.float32)  # (22, 6)
SET_INDEX = {y: i for i, y in enumerate(SETS)}


def report(y: tuple, variant: int) -> str:
    """'enlarged heart. pleural effusion. no pneumothorax.' from a multi-hot label tuple and a phrasing variant."""
    found = [f for f in range(6) if y[f]]
    if not found:
        return NORMAL[variant & 1]
    parts = [PHRASES[f][(variant >> i) & 1] for i, f in enumerate(found)]
    if not y[4] and variant & 4:
        parts.append(NEGATION)
    return " ".join(parts)


TEMPLATES = {report(y, v): y for y in SETS for v in range(N_VARIANTS)}


def parse_report(text: str) -> tuple:
    """Labels back from any text by keyword (negated pneumothorax excluded): the multi-hot tuple (6,)."""
    t = text.lower()
    return tuple(int(re.search(k, t) is not None) for k in KEYWORDS)


def template_labels(text: str) -> tuple | None:
    """The labels if the text is exactly one of the template reports, else None (not well formed)."""
    return TEMPLATES.get(text)


def header(y: tuple) -> str:
    return "".join(c if v else "-" for c, v in zip(CODES, y))


def sequence(y: tuple, variant: int) -> str:
    """'ce----' + '>' + report + '|' padded with '_' to CTX characters."""
    return (header(y) + BOS + report(y, variant) + EOS).ljust(CTX, PAD)


def encode(s: str) -> list[int]:
    return [STOI[c] for c in s]


def decode(ids) -> str:
    return "".join(VOCAB[int(i)] for i in ids)


def report_text(ids) -> str:
    """The text a sequence writes after BOS, up to EOS (or the end)."""
    s = decode(ids)[K + 1 :]
    return s.split(EOS)[0].replace(PAD, "").strip()


def target_mask(tokens: torch.Tensor) -> torch.Tensor:
    """Targets j ≥ K (the first letter is predicted at the BOS position) up to and including EOS: (…, T − 1)."""
    tgt = tokens[..., 1:]
    j = torch.arange(tgt.shape[-1], device=tokens.device)
    after_eos = (tgt == STOI[EOS]).long().cumsum(-1) - (tgt == STOI[EOS]).long()  # 1 strictly after the first EOS
    return (j >= K) & (after_eos == 0) & (tgt != STOI[PAD])


def template_table() -> dict:
    """Every template sequence, row = set_id · N_VARIANTS + variant: tokens (176, CTX), mask (176, CTX − 1)."""
    tok = torch.tensor([encode(sequence(y, v)) for y in SETS for v in range(N_VARIANTS)])
    return {"tokens": tok, "mask": target_mask(tok)}


def row_of(set_ids: torch.Tensor, variants: torch.Tensor) -> torch.Tensor:
    return set_ids * N_VARIANTS + variants


def clean_text(s: str, max_chars: int = MAX_CAP) -> tuple[str, bool]:
    """Lower case, without figure labels, '(Fig. 2)'-style references and panel letters, letters outside the
    vocabulary as spaces; then whole sentences while they fit in max_chars. Only when the first sentence alone is
    longer is it cut at a word boundary (returned flag True)."""
    t = s.lower()
    t = re.sub(r"^\s*(fig(ure)?\.?\s*\d+[a-z]?\s*[.:]?\s*)", "", t)
    t = re.sub(r"[(\[]\s*(see\s+)?(fig|figure|figs)\b[^)\]]*[)\]]", " ", t)
    t = re.sub(r"\(\s*[a-z]\s*\)|^\s*[a-z]\s*[:.)]\s+", " ", t)
    t = "".join(c if c in STOI and c not in (PAD, IMG, BOS, EOS) else " " for c in t)
    t = re.sub(r"\s+", " ", t).strip()
    t = re.sub(r"\s+([.,])", r"\1", t)
    out = ""
    for sent in re.split(r"(?<=\.)\s+", t):
        sent = sent if sent.endswith(".") else sent + "."
        if len(out) + len(sent) + (1 if out else 0) > max_chars:
            break
        out = f"{out} {sent}" if out else sent
    if out:
        return out, False
    return t[: max_chars + 1].rsplit(" ", 1)[0].rstrip(" ,.-"), True  # one over-long first sentence


def caption_tokens(texts: list[str]) -> dict:
    """Stage-2 sequences: a blank header '------' (replaced by the soft prefix), BOS, text, EOS: (N, CTX_CAP)."""
    tok = torch.tensor([encode(("-" * K + BOS + c + EOS).ljust(CTX_CAP, PAD)) for c in texts])
    return {"tokens": tok, "mask": target_mask(tok)}


# ---------------------------------------------------------------- data
def parse_nih(s: str) -> tuple:
    """'Cardiomegaly|Effusion' → (1, 1, 0, 0, 0, 0); 'No Finding' → zeros; Mass and Nodule share one finding."""
    y = [0] * 6
    for name in s.split("|"):
        if name != "No Finding":
            y[NIH_NAMES[name]] = 1
    return tuple(y)


def _standardise(images: np.ndarray, device) -> torch.Tensor:
    """uint8 (N, H, W) → float, each image to zero mean and unit variance (the two sources have different
    exposure and contrast)."""
    x = torch.from_numpy(images).float()
    x = (x - x.mean((1, 2), keepdim=True)) / (x.std((1, 2), keepdim=True) + 1e-3)
    return x.to(device)


def load_nih(device) -> dict:
    """x (N, 64, 64) standardised, y (N, 6) multi-hot float, sets (N,) index into SETS; split 0 train / 1 test."""
    with np.load(DATA / "nih_cxr_subset.npz") as z:  # images (N,64,64) uint8, labels (N,), findings (N,6), split
        images, findings, split = z["images"], z["findings"], z["split"]
    out = {"blank": torch.zeros(IMG_SIZE, IMG_SIZE, device=device)}
    for name, s in (("train", 0), ("test", 1)):
        y = torch.from_numpy(findings[split == s].astype(np.float32))
        out[f"x_{name}"] = _standardise(images[split == s], device)
        out[f"y_{name}"] = y.to(device)
        out[f"sets_{name}"] = torch.tensor([SET_INDEX[tuple(int(v) for v in r)] for r in y.tolist()], device=device)
    return out


def load_pmc(device) -> dict:
    """PMC figures x (N, 64, 64) standardised and their stage-2 texts (caption + the article's sentences that cite
    the figure, cleaned, whole sentences up to MAX_CAP characters); also the caption alone, for comparison."""
    with np.load(DATA / "pmc_cxr_subset.npz") as z:  # images (N,64,64) uint8, labels (N,), split (N,)
        images, split = z["images"], z["split"]
    with open(DATA / "pmc_captions.csv", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    caps = [clean_text(r["report_text"])[0] for r in rows]
    only = [clean_text(r["caption"])[0] for r in rows]
    out = {"cut": sum(clean_text(r["report_text"])[1] for r in rows)}
    for name, s in (("train", 0), ("test", 1)):
        out[f"x_{name}"] = _standardise(images[split == s], device)
        out[f"cap_{name}"] = [c for c, sp in zip(caps, split) if sp == s]
        out[f"caponly_{name}"] = [c for c, sp in zip(only, split) if sp == s]
        out[f"tok_{name}"] = {k: v.to(device) for k, v in caption_tokens(out[f"cap_{name}"]).items()}
    return out


# ---------------------------------------------------------------- stacked weights (model axis first)
def init_cnn(seeds: list[int], salt: int, device, emb: int = 64) -> dict:
    """conv 5×5 1→8, pool, conv 3×3 8→16, pool, conv 3×3 16→16, pool (64 → 6×6), 576 → emb features → 6 logits."""

    def make(g):
        def u(*shape, fan):
            return (torch.rand(*shape, generator=g) * 2 - 1) * math.sqrt(6 / fan)  # He-uniform

        return {"c1": u(8, 1, 5, 5, fan=25), "b1": torch.zeros(8), "c2": u(16, 8, 3, 3, fan=72),
                "b2": torch.zeros(16), "c3": u(16, 16, 3, 3, fan=144), "b3": torch.zeros(16),
                "fc": u(576, emb, fan=576), "bf": torch.zeros(emb), "head": u(emb, 6, fan=emb) / 2,
                "bh": torch.zeros(6)}  # fmt: skip

    return _stack(seeds, salt, make, device)


def init_lm(seeds: list[int], salt: int, device, d: int = 32, heads: int = 4, layers: int = 2) -> dict:
    """The tiny GPT of chain_of_thought / stitch_llm_cnn_dag, with this problem's vocabulary and context."""
    v, dk, s = len(VOCAB), d // heads, 1 / math.sqrt(d)

    def make(g):
        def r(*shape, scale):
            return scale * torch.randn(*shape, generator=g)

        def layer():
            return {"g1": torch.ones(d), "b1": torch.zeros(d), "Wq": r(heads, d, dk, scale=s),
                    "bq": torch.zeros(heads, dk), "Wk": r(heads, d, dk, scale=s), "bk": torch.zeros(heads, dk),
                    "Wv": r(heads, d, dk, scale=s), "bv": torch.zeros(heads, dk),
                    "Wo": r(heads, dk, d, scale=s / math.sqrt(2 * layers)), "bo": torch.zeros(d),
                    "g2": torch.ones(d), "b2": torch.zeros(d), "W1": r(d, 4 * d, scale=s), "c1": torch.zeros(4 * d),
                    "W2": r(4 * d, d, scale=0.5 * s / math.sqrt(2 * layers)), "c2": torch.zeros(d)}  # fmt: skip

        return {"tok": r(v, d, scale=0.3), "pos": r(CTX_CAP, d, scale=0.3), "layers": [layer() for _ in range(layers)],
                "gf": torch.ones(d), "bf": torch.zeros(d), "head": r(d, v, scale=1e-3),
                "bias": torch.zeros(v)}  # fmt: skip

    return _stack(seeds, salt, make, device)


def init_connector(seeds: list[int], salt: int, device, d: int = 32, emb: int = 6, hidden: int = 64) -> dict:
    """c_t(q) = GELU(q W1 + S_t) W2 + b2: the CNN's six finding probabilities q and a slot vector S_t → an LM input
    vector at header slot t."""

    def make(g):
        return {"W1": torch.randn(emb, hidden, generator=g) / math.sqrt(emb),
                "S": 0.1 * torch.randn(K, hidden, generator=g),
                "W2": torch.randn(hidden, d, generator=g) * 0.3 / math.sqrt(hidden), "b2": torch.zeros(d)}  # fmt: skip

    return _stack(seeds, salt, make, device)


def init_router(seeds: list[int], salt: int, device) -> dict:
    """Linear router over the position features (8×8 pooled image ⊕ one-hot id) → 2 expert logits."""
    f = 64 + len(VOCAB)

    def make(g):
        return {"W": 0.01 * torch.randn(f, 2, generator=g), "b": torch.zeros(2)}

    return _stack(seeds, salt, make, device)


def cat_models(ws: list):
    """Several stacks of the same architecture joined along the model axis (to run them in ONE batched call)."""
    if isinstance(ws[0], dict):
        return {k: cat_models([w[k] for w in ws]) for k in ws[0]}
    if isinstance(ws[0], list):
        return [cat_models([w[i] for w in ws]) for i in range(len(ws[0]))]
    return torch.cat([w.detach() for w in ws])


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/ai/cxr_report_dag/solve.py
"""The training process as a tensor DAG: every rule of data/training.smk is ONE batched call over the seeds.

L0  train_cnn ∥ train_lm ∥ train_monolith   (separate pretraining on NIH labels and on template reports; the
                                             monolith baseline is just another rule)
L1  train_connector                          (CNN and LM frozen, only the connector learns, NIH image → report)
L2  global_finetune ∥ train_router           (everything end to end; or a learned router on the frozen experts)
L3  caption_finetune                         (stage 2: connector + LM on real PMC figure captions, CNN frozen)
L4  evaluate

The DAG algebra (Kronecker job graph, boolean levels, max-plus finish times, critical path) is imported from the
Circle of Willis workflow problem, the optimiser loop from problems/ai/stitch_llm_cnn_dag.
"""

import time
from dataclasses import dataclass, field

import torch
import torch.nn.functional as F

from problems.ai.stitch_llm_cnn_dag.solve import fit, rule_graph
from problems.medical.cow_workflow_dag import equation as dag
from problems.medical.cow_workflow_dag.solve import parse_snakefile

from . import equation as eq
from . import tensors as T


@dataclass
class Config:
    seeds: list[int] = field(default_factory=lambda: [0, 1])
    batch: int = 32
    steps_cnn: int = 250
    steps_lm: int = 200
    steps_con: int = 120
    steps_ft: int = 60
    steps_router: int = 40
    steps_cap: int = 150
    d: int = 32
    heads: int = 4
    layers: int = 2
    lr: float = 3e-3  # CNN and router
    lr_lm: float = 1e-2
    lr_con: float = 3e-2
    lr_ft: float = 3e-4
    lr_cap: float = 1e-2
    batch_cap: int = 16  # stage-2 sequences are about six times longer
    lr_mono: float = 2e-3
    shift: int = 3  # random translation of the training images (pixels)
    n_test: int | None = None  # evaluate on the first n_test test images only (fast mode)

    def macs(self) -> dict:
        """Multiply-adds of one forward pass per example (backward ≈ 2× forward for every one, so ratios hold)."""
        d, t, v = self.d, T.CTX - 1, len(T.VOCAB)
        cnn = 60 * 60 * 8 * 25 + 28 * 28 * 16 * 72 + 12 * 12 * 16 * 144 + 576 * 64 + 64 * 6
        lm = t * (self.layers * (12 * d * d + 2 * t * d) + d * v)
        con = T.K * (6 * 64 + 64 * d)
        return {"cnn": cnn, "lm": lm, "joint": cnn + con + lm}

    @property
    def steps_mono(self) -> int:
        """Matched compute: the monolith's joint steps cost as many multiply-adds as the DAG pipeline's CNN, LM,
        connector and fine-tune steps together (the caption stage is not part of the stage-1 comparison)."""
        c = self.macs()
        work = self.steps_cnn * c["cnn"] + self.steps_lm * c["lm"] + (self.steps_con + self.steps_ft) * c["joint"]
        return round(work / c["joint"])


# ---------------------------------------------------------------- the rule graph (parsed with the CoW parser)
def build_dag(cfg: Config, text: str | None = None) -> dict:
    """Rules, A_rule, rule levels and the (J, J) job adjacency A = Σ A_rule ⊗ (Kronecker blocks over the seeds)."""
    rules = parse_snakefile(text if text is not None else T.TRAINING_GRAPH.read_text(encoding="utf-8"))
    A_rule, axes = rule_graph(rules)
    sizes = {"seed": len(cfg.seeds)}
    A, off = dag.job_adjacency(A_rule, axes, sizes)
    return {"rules": rules, "names": [r["name"] for r in rules], "A_rule": A_rule, "axes": axes, "sizes": sizes,
            "levels": dag.levels(A_rule), "A": A, "off": off}  # fmt: skip


def schedule(d: dict, seconds: dict) -> dict:
    """Max-plus finish times with the measured rule durations: with one worker per rule the makespan is the
    critical path; one worker runs the sum."""
    dur = torch.tensor([seconds[n] for n in d["names"]], dtype=torch.float64)
    t = dag.finish_times(d["A_rule"].double(), dur)
    path = dag.critical_path(d["A_rule"].double(), t, dur)
    return {"finish": t, "duration": dur, "path": [d["names"][i] for i in path], "makespan": float(t.max()),
            "serial": float(dur.sum())}  # fmt: skip


# ---------------------------------------------------------------- batches
def shifted(x: torch.Tensor, seeds: list[int], step: int, salt: int, shift: int) -> torch.Tensor:
    """Random integer translations of each image (M, B, H, W), zero padded, drawn per model from its own generator
    (so a model's augmentation never depends on which other models share the stack)."""
    if shift == 0:
        return x
    m, b, h, w = x.shape
    off = torch.stack([torch.randint(-shift, shift + 1, (b, 2), generator=torch.Generator().manual_seed(
        s * 104_729 + step * 31 + salt)) for s in seeds]).to(x.device)  # fmt: skip
    pad = F.pad(x, (shift, shift, shift, shift))
    iy = (torch.arange(h, device=x.device) + shift)[None, None, :] - off[..., 0:1]  # (M, B, H)
    ix = (torch.arange(w, device=x.device) + shift)[None, None, :] - off[..., 1:2]
    rows = pad.gather(2, iy[..., None].expand(m, b, h, w + 2 * shift))
    return rows.gather(3, ix[:, :, None, :].expand(m, b, h, w))


def joint_batch(st: dict, rows: torch.Tensor, variants: torch.Tensor | None = None, train: bool = True) -> dict:
    """NIH images rows (M, B) with their template reports (phrasing variant per example) → x, tokens, mask."""
    d, tab = st["data"], st["table"]
    sets = (d["sets_train"] if train else d["sets_test"])[rows]
    v = rows % T.N_VARIANTS if variants is None else variants
    tok = tab["tokens"][T.row_of(sets, v)]
    return {"x": (d["x_train"] if train else d["x_test"])[rows], "tok": tok, "mask": tab["mask"][T.row_of(sets, v)]}


def joint_loss(st: dict, rows, w_cnn, w_con, w_lm, variants=None, step: int = 0, salt: int = 0) -> torch.Tensor:
    """Mean −log p of the report letters given the image (the header is the soft prefix).  → (M,) nats"""
    bt = joint_batch(st, rows, variants)
    x = shifted(bt["x"], st["cfg"].seeds, step, salt, st["cfg"].shift) if salt else bt["x"]
    z = eq.stitched_logits(x, bt["tok"][..., :-1], w_cnn, w_con, w_lm)
    return eq.masked_ce(z, bt["tok"][..., 1:], bt["mask"])


def moe_logits(x, tok, w_cnn, w_con, w_lm, w_router, blank, soft: bool = False, hard: bool = False, q_img=None):
    """LM logits through the MoE input layer: fixed router (w_router None) or a learned one; hard = route then run
    (G one-hot, the chosen expert's output not scaled by p), as used for writing."""
    ids, is_img = eq.position_items(tok)
    m, n, _ = ids.shape
    if w_router is None:
        G = p = eq.route_fixed(m, n, is_img)
    else:
        G, p = eq.route_learned(eq.router_features(x, ids, is_img, blank), w_router, soft)
        G = (G > 0).to(p.dtype) if hard else G
    return eq.lm_logits(eq.moe_inputs(x, ids, G, w_cnn, w_con, w_lm, blank, q_img), w_lm), G, p


# ---------------------------------------------------------------- the rules: one batched call each
def rule_train_cnn(st: dict) -> dict:
    cfg, d, S = st["cfg"], st["data"], len(st["cfg"].seeds)
    w = T.init_cnn(cfg.seeds, 1, st["dev"])
    rows = T.draws(cfg.seeds, [len(d["y_train"])] * S, cfg.steps_cnn, cfg.batch, 1, st["dev"])
    pos = d["y_train"].mean(0)
    pw = ((1 - pos) / pos).clamp(max=8.0)

    def loss(t):
        x = shifted(d["x_train"][rows[t]], cfg.seeds, t, 1, cfg.shift)
        return eq.bce(eq.cnn_logits(eq.cnn_features(x, w), w), d["y_train"][rows[t]], pw)

    return {"w": w, "loss": fit(T.leaves(w), loss, cfg.steps_cnn, cfg.lr, "train_cnn")}


def rule_train_lm(st: dict) -> dict:
    """The LM alone on every template report, the true label header written as text in front."""
    cfg, tab, S = st["cfg"], st["table"], len(st["cfg"].seeds)
    w = T.init_lm(cfg.seeds, 2, st["dev"], cfg.d, cfg.heads, cfg.layers)
    rows = T.draws(cfg.seeds, [len(tab["tokens"])] * S, cfg.steps_lm, cfg.batch, 2, st["dev"])

    def loss(t):
        x = tab["tokens"][rows[t]]
        return eq.masked_ce(eq.lm_logits(eq.embed(x[..., :-1], w), w), x[..., 1:], tab["mask"][rows[t]])

    return {"w": w, "loss": fit(T.leaves(w), loss, cfg.steps_lm, cfg.lr_lm, "train_lm")}


def _joint_draws(st: dict, steps: int, salt: int):
    cfg, n = st["cfg"], len(st["data"]["y_train"])
    S = len(cfg.seeds)
    rows = T.draws(cfg.seeds, [n] * S, steps, cfg.batch, salt, st["dev"])
    var = T.draws(cfg.seeds, [T.N_VARIANTS] * S, steps, cfg.batch, salt + 100, st["dev"])
    return rows, var


def rule_train_monolith(st: dict) -> dict:
    """The non-DAG baseline: the same CNN + connector + LM, random init, trained end to end on image → report only."""
    cfg, dev = st["cfg"], st["dev"]
    w_cnn, w_con = T.init_cnn(cfg.seeds, 11, dev), T.init_connector(cfg.seeds, 13, dev, cfg.d)
    w_lm = T.init_lm(cfg.seeds, 12, dev, cfg.d, cfg.heads, cfg.layers)
    rows, var = _joint_draws(st, cfg.steps_mono, 5)
    params = T.leaves(w_cnn) + T.leaves(w_con) + T.leaves(w_lm)
    hist = fit(params, lambda t: joint_loss(st, rows[t], w_cnn, w_con, w_lm, var[t], t, 5), cfg.steps_mono,
               cfg.lr_mono, "train_monolith")  # fmt: skip
    return {"cnn": w_cnn, "con": w_con, "lm": w_lm, "loss": hist}


def rule_train_connector(st: dict) -> dict:
    """Freeze the pretrained CNN and LM; only the connector learns (NIH image → template report)."""
    cfg, dev = st["cfg"], st["dev"]
    w_cnn, w_lm = T.clone(st["train_cnn"]["w"], False), T.clone(st["train_lm"]["w"], False)
    w_con = T.init_connector(cfg.seeds, 3, dev, cfg.d)
    init = T.clone(w_con, False)
    rows, var = _joint_draws(st, cfg.steps_con, 3)
    hist = fit(T.leaves(w_con), lambda t: joint_loss(st, rows[t], w_cnn, w_con, w_lm, var[t], t, 3), cfg.steps_con,
               cfg.lr_con, "train_connector")  # fmt: skip
    return {"cnn": w_cnn, "con": w_con, "lm": w_lm, "con_init": init, "loss": hist}


def rule_global_finetune(st: dict) -> dict:
    """Unfreeze everything and train end to end through the DAG edge, gently."""
    cfg, c = st["cfg"], st["train_connector"]
    w_cnn, w_con, w_lm = (T.clone(c[k], True) for k in ("cnn", "con", "lm"))
    rows, var = _joint_draws(st, cfg.steps_ft, 4)
    params = T.leaves(w_cnn) + T.leaves(w_con) + T.leaves(w_lm)
    hist = fit(params, lambda t: joint_loss(st, rows[t], w_cnn, w_con, w_lm, var[t], t, 4), cfg.steps_ft, cfg.lr_ft,
               "global_finetune")  # fmt: skip
    return {"cnn": w_cnn, "con": w_con, "lm": w_lm, "loss": hist}


def rule_train_router(st: dict) -> dict:
    """Replace the fixed modality routing by a learned linear router (experts frozen), trained as a dense mixture
    G = p (both experts everywhere, so the gradient compares them) and used top-1: does it find the routing again?"""
    cfg, c, blank = st["cfg"], st["train_connector"], st["data"]["blank"]
    w_r = T.init_router(cfg.seeds, 4, st["dev"])
    rows, var = _joint_draws(st, cfg.steps_router, 6)

    def loss(t):
        bt = joint_batch(st, rows[t], var[t])
        z, _, _ = moe_logits(bt["x"], bt["tok"][..., :-1], c["cnn"], c["con"], c["lm"], w_r, blank, soft=True)
        return eq.masked_ce(z, bt["tok"][..., 1:], bt["mask"])

    return {"router": w_r, "loss": fit(T.leaves(w_r), loss, cfg.steps_router, cfg.lr, "train_router")}


def rule_caption_finetune(st: dict) -> dict:
    """Stage 2: the stitched model (after the global fine-tune) learns real figure captions; CNN frozen."""
    cfg, ft, p = st["cfg"], st["global_finetune"], st["pmc"]
    w_cnn = T.clone(ft["cnn"], False)
    w_con, w_lm = T.clone(ft["con"], True), T.clone(ft["lm"], True)
    rows = T.draws(cfg.seeds, [len(p["cap_train"])] * len(cfg.seeds), cfg.steps_cap, cfg.batch_cap, 8, st["dev"])
    tok, mask = p["tok_train"]["tokens"], p["tok_train"]["mask"]

    def loss(t):
        x = shifted(p["x_train"][rows[t]], cfg.seeds, t, 8, cfg.shift)
        z = eq.stitched_logits(x, tok[rows[t]][..., :-1], w_cnn, w_con, w_lm)
        return eq.masked_ce(z, tok[rows[t]][..., 1:], mask[rows[t]])

    hist = fit(T.leaves(w_con) + T.leaves(w_lm), loss, cfg.steps_cap, cfg.lr_cap, "caption_finetune")
    return {"cnn": w_cnn, "con": w_con, "lm": w_lm, "loss": hist}


# ---------------------------------------------------------------- greedy decoding
def blank_prompt(m: int, n: int, headers: list[list[str]] | None = None, device=None) -> torch.Tensor:
    """(M, N, CTX) ids: a header ('------', or headers[m][n]) + BOS, then padding to be written over."""
    hs = headers or [["-" * T.K] * n] * m
    return torch.tensor([[T.encode((h + T.BOS).ljust(T.CTX, T.PAD)) for h in row] for row in hs], device=device)


@torch.no_grad()
def _greedy(logits_fn, tok: torch.Tensor, n_new: int) -> torch.Tensor:
    """Write n_new letters after BOS, each the argmax given what is written so far (causal: only the prefix is fed);
    stop early once every row has written EOS (later letters stay padding)."""
    done = torch.zeros(tok.shape[:-1], dtype=torch.bool, device=tok.device)
    for k in range(n_new):
        p = T.K + 1 + k
        nxt = logits_fn(tok[..., :p])[..., p - 1, :].argmax(-1)
        tok[..., p] = torch.where(done, torch.full_like(nxt, T.STOI[T.PAD]), nxt)
        done |= nxt == T.STOI[T.EOS]
        if bool(done.all()):
            break
    return tok


def _prompt_ids(m: int, n: int, device) -> torch.Tensor:
    return torch.tensor(T.encode("-" * T.K + T.BOS), device=device).expand(m, n, -1)


def generate(x, w_cnn, w_con, w_lm, n_new: int = T.MAX_TXT + 1) -> torch.Tensor:
    """The stitched model writes from the image alone (soft header from the CNN, then BOS), with a key/value cache.
    x (M, N, 64, 64) → ids (M, N, K + 1 + n_new)."""
    m, n = x.shape[:2]
    bos = eq.embed(torch.full((m, n, 1), T.STOI[T.BOS], device=x.device), w_lm)
    gen = eq.greedy_cached(torch.cat([eq.soft_prefix(x, w_cnn, w_con), bos], 2), w_lm, n_new)
    return torch.cat([_prompt_ids(m, n, x.device), gen], -1)


def generate_moe(x, w_cnn, w_con, w_lm, w_router, blank, n_new: int = T.MAX_TXT + 1) -> torch.Tensor:
    """Through the MoE input layer (learned router, route then run): every letter re-routes the whole prefix, so no
    cache here."""
    tok = blank_prompt(x.shape[0], x.shape[1], device=x.device)
    q = eq.cnn_output(x, w_cnn)
    return _greedy(lambda t: moe_logits(x, t, w_cnn, w_con, w_lm, w_router, blank, hard=True, q_img=q)[0], tok, n_new)


def generate_text(tok: torch.Tensor, w_lm, n_new: int = T.MAX_TXT + 1) -> torch.Tensor:
    """The LM alone, given a written header + BOS (the first K + 1 ids of tok).  → ids (M, N, K + 1 + n_new)."""
    gen = eq.greedy_cached(eq.embed(tok[..., : T.K + 1], w_lm), w_lm, n_new)
    return torch.cat([tok[..., : T.K + 1], gen], -1)


def texts(tok: torch.Tensor) -> list[list[str]]:
    return [[T.report_text(r) for r in m] for m in tok]


# ---------------------------------------------------------------- evaluation
def report_scores(txt: list[list[str]], y_true: torch.Tensor, sets_true: torch.Tensor) -> dict:
    """Parse generated reports back into labels: micro and macro F1, share of well-formed templates, and share of
    exact templates of the true label set.  Per model (M,)."""
    pred = torch.tensor([[T.parse_report(s) for s in m] for m in txt], dtype=torch.bool)
    micro, macro = eq.f1(pred, y_true.bool().cpu())
    lab = [[T.template_labels(s) for s in m] for m in txt]
    true = [T.SETS[int(i)] for i in sets_true]
    wf = torch.tensor([[a is not None for a in m] for m in lab]).float().mean(1)
    ex = torch.tensor([[a == b for a, b in zip(m, true)] for m in lab]).float().mean(1)
    return {"micro": micro, "macro": macro, "wellformed": wf, "exact": ex, "pred": pred}


def caption_scores(gen: list[list[str]], ref: list[str]) -> dict:
    """BLEU-1, ROUGE-L and finding-keyword recall/precision of generated captions against the true ones (M,)."""
    out = {k: [] for k in ("bleu1", "rouge_l", "kw_recall", "kw_precision")}
    yt = torch.tensor([T.parse_report(r) for r in ref], dtype=torch.float32)
    for m in gen:
        out["bleu1"].append(sum(eq.bleu1(h, r) for h, r in zip(m, ref)) / len(ref))
        out["rouge_l"].append(sum(eq.rouge_l(h, r) for h, r in zip(m, ref)) / len(ref))
        yp = torch.tensor([T.parse_report(h) for h in m], dtype=torch.float32)
        tp = (yp * yt).sum()
        out["kw_recall"].append(float(tp / yt.sum().clamp(min=1)))
        out["kw_precision"].append(float(tp / yp.sum().clamp(min=1)))
    return {k: torch.tensor(v) for k, v in out.items()}


def constant_caption(captions: list[str], n: int = 40) -> str:
    """A blind baseline: of n training texts drawn at random (seed 0), the one with the highest mean ROUGE-L against
    n other training texts."""
    g = torch.Generator().manual_seed(0)
    idx = torch.randperm(len(captions), generator=g)[: 2 * n].tolist()
    cand, ref = [captions[i] for i in idx[:n]], [captions[i] for i in idx[n:]]
    return max(cand, key=lambda c: sum(eq.rouge_l(c, r) for r in ref))


@torch.no_grad()
def rule_evaluate(st: dict) -> dict:
    """Every stage writes a report for every NIH test image; the stage-1 stacks are joined on the model axis and
    decoded in ONE batched call. Then stage 2: captions for the held-out PMC figures."""
    cfg, dev = st["cfg"], st["dev"]
    d, p = dict(st["data"]), dict(st["pmc"])
    if cfg.n_test:
        for k in ("x_test", "y_test", "sets_test"):
            d[k] = d[k][: cfg.n_test]
        p["x_test"], p["cap_test"] = p["x_test"][: cfg.n_test], p["cap_test"][: cfg.n_test]
    S = len(cfg.seeds)
    con, ft, mono = st["train_connector"], st["global_finetune"], st["train_monolith"]
    stages = {"connector": (con["cnn"], con["con"], con["lm"]), "finetune": (ft["cnn"], ft["con"], ft["lm"]),
              "monolith": (mono["cnn"], mono["con"], mono["lm"])}  # fmt: skip
    w = [T.cat_models([v[i] for v in stages.values()]) for i in range(3)]
    xt = d["x_test"][None].expand(S * len(stages), -1, -1, -1)
    out = generate(xt, *w)
    gen = {k: out[i * S : (i + 1) * S] for i, k in enumerate(stages)}
    gen["router"] = generate_moe(xt[:S], con["cnn"], con["con"], con["lm"], st["train_router"]["router"], d["blank"])
    # unimodal: the CNN alone (threshold 0.5) and its labels written as a text header for the LM alone (symbolic)
    q = eq.cnn_output(xt[:S], st["train_cnn"]["w"])  # (S, N, 6)
    cnn_pred = q > 0.5
    heads = [[T.header(tuple(int(v) for v in r)) for r in m] for m in cnn_pred.tolist()]
    gen["symbolic"] = generate_text(blank_prompt(S, len(d["y_test"]), heads, dev), st["train_lm"]["w"])
    txt = {k: texts(v) for k, v in gen.items()}
    rep = {k: report_scores(v, d["y_test"], d["sets_test"]) for k, v in txt.items()}
    cnn_micro, cnn_macro = eq.f1(cnn_pred.cpu(), d["y_test"].bool().cpu())
    auc = torch.stack([torch.stack([eq.auroc(q[m, :, f].cpu(), d["y_test"][:, f].cpu()) for f in range(6)])
                       for m in range(S)])  # fmt: skip
    # random guessing at the training prevalence: expected counts → expected micro F1
    pt, pi = d["y_train"].mean(0).cpu(), d["y_test"].mean(0).cpu()
    tp, fp, fn = (pt * pi).sum(), (pt * (1 - pi)).sum(), ((1 - pt) * pi).sum()
    chance = float(2 * tp / (2 * tp + fp + fn))
    # the learned router: share of positions sent to the modality's expert, and its routing on the written reports
    tok0 = gen["router"][..., :-1]
    ids, is_img = eq.position_items(tok0)
    _, G, prob = moe_logits(xt[:S], tok0, con["cnn"], con["con"], con["lm"], st["train_router"]["router"], d["blank"])
    route_acc = ((G[..., 0] > 0) == is_img).float().mean((1, 2)).cpu()

    # stage 2: captions of held-out PMC figures (before and after the caption stage; with shuffled images)
    n_p = len(p["cap_test"])
    perm = torch.roll(torch.arange(n_p, device=dev), 1)
    cap = st["caption_finetune"]
    xp = p["x_test"][None].expand(S, -1, -1, -1)
    w2 = [T.cat_models([a[k], b[k], b[k]]) for k, a, b in zip(("cnn", "con", "lm"), (ft,) * 3, (cap,) * 3)]
    out2 = generate(torch.cat([xp, xp, xp[:, perm]]), *w2, n_new=T.MAX_CAP + 1)
    cgen = {k: out2[i * S : (i + 1) * S] for i, k in enumerate(("finetune", "caption", "caption_shuffled"))}
    ctxt = {k: texts(v) for k, v in cgen.items()}
    const = constant_caption(p["cap_train"])
    ctxt["constant"] = [[const] * n_p] * S
    cscore = {k: caption_scores(v, p["cap_test"]) for k, v in ctxt.items()}
    # held-out cross-entropy of the true texts (teacher forcing, nats per letter): does the image help at all?
    tok = st["pmc"]["tok_test"]["tokens"][:n_p][None].expand(3 * S, -1, -1)
    msk = st["pmc"]["tok_test"]["mask"][:n_p][None].expand(3 * S, -1, -1)
    x3, num, den = torch.cat([xp, xp, xp[:, perm]]), 0.0, 0.0
    for lo in range(0, n_p, 8):  # chunks: the (M, N, H, T, T) attention maps of 367 letters are large
        sl = slice(lo, lo + 8)
        n_mask = msk[:, sl].sum((1, 2))
        num = num + n_mask * eq.masked_ce(eq.stitched_logits(x3[:, sl], tok[:, sl, :-1], *w2), tok[:, sl, 1:],
                                          msk[:, sl])  # fmt: skip
        den = den + n_mask
    ce = num / den
    for i, k in enumerate(("finetune", "caption", "caption_shuffled")):
        cscore[k]["nats"] = ce[i * S : (i + 1) * S].cpu()
    cscore["constant"]["nats"] = torch.full((S,), float("nan"))
    return {"txt": txt, "rep": rep, "cnn_micro": cnn_micro, "cnn_macro": cnn_macro, "auc": auc, "q": q.cpu(),
            "chance": chance, "route_acc": route_acc, "route_p": prob[..., 0].cpu(), "cap_txt": ctxt, "cap": cscore,
            "constant": const}  # fmt: skip


RULES = {"train_cnn": rule_train_cnn, "train_lm": rule_train_lm, "train_monolith": rule_train_monolith,
         "train_connector": rule_train_connector, "global_finetune": rule_global_finetune,
         "train_router": rule_train_router, "caption_finetune": rule_caption_finetune,
         "evaluate": rule_evaluate}  # fmt: skip


def make_store(cfg: Config, dev: torch.device) -> dict:
    """Data and tables every rule reads."""
    return {"cfg": cfg, "dev": dev, "data": T.load_nih(dev), "pmc": T.load_pmc(dev),
            "table": {k: v.to(dev) for k, v in T.template_table().items()}}  # fmt: skip


def run_engine(cfg: Config, dev: torch.device, d: dict | None = None) -> tuple[dict, dict, dict]:
    """Run the training DAG: rules level by level (levels of A_rule), each one batched call over the seeds."""
    d = d or build_dag(cfg)
    st = make_store(cfg, dev)
    seconds = {}
    for k in range(int(d["levels"].max()) + 1):
        for r in (d["levels"] == k).nonzero().flatten().tolist():
            name = d["names"][r]
            t0 = time.perf_counter()
            st[name] = RULES[name](st)
            seconds[name] = time.perf_counter() - t0
    return st, seconds, d


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/ai/cxr_report_dag/plot.py
"""Animate the experiment: the training DAG runs level by level while the per-stage loss curves grow; then the label
F1 of every stage against the monolith (seed dots), the caption metrics, the router G (fixed vs learned), and test
X-rays with the CNN's finding bars and the report typed out letter by letter, and PMC figures with their captions.
A static PNG holds the measured schedule (serial vs critical path).
Tables: stages.csv, captions.csv, caption_examples.csv, report_examples.csv and timings.csv."""

import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

from problems._common.plotting import ACCENT, save_png
from problems.ai.stitch_llm_cnn_dag.plot import save_gif_shared_palette

from . import tensors as T

STAGE = {  # key: (label, colour)
    "cnn": ("CNN alone", ACCENT[5]),
    "symbolic": ("CNN label→LM", "#6b7280"),
    "connector": ("L1 connector", ACCENT[0]),
    "finetune": ("L2 fine-tune", ACCENT[2]),
    "router": ("learned router", ACCENT[6]),
    "monolith": ("monolith", ACCENT[1]),
}
CAP = {"finetune": ("stage 1 only", ACCENT[2]), "caption": ("L3 captions", ACCENT[3]),
       "caption_shuffled": ("shuffled img.", "#9ca3af"), "constant": ("constant", "#374151")}  # fmt: skip
RULE_COLOUR = {"train_cnn": ACCENT[5], "train_lm": ACCENT[4], "train_monolith": ACCENT[1],
               "train_connector": ACCENT[0], "global_finetune": ACCENT[2], "train_router": ACCENT[6],
               "caption_finetune": ACCENT[3], "evaluate": "#374151"}  # fmt: skip
SHORT = {"train_cnn": "CNN", "train_lm": "LM", "train_monolith": "monolith", "train_connector": "conn.",
         "global_finetune": "fine-tune", "train_router": "router", "caption_finetune": "captions",
         "evaluate": "eval"}  # fmt: skip


def _node_xy(res: dict) -> dict:
    xy, lev = {}, res["levels"]
    for k in range(max(lev) + 1):
        group = [n for n, lv in zip(res["names"], lev) if lv == k]
        for i, n in enumerate(group):
            xy[n] = (k, (len(group) - 1) / 2 - i)
    return xy


def _draw_dag(ax, res: dict, done_level: float) -> None:
    ax.clear()
    xy = _node_xy(res)
    path = set(zip(res["path"][:-1], res["path"][1:]))
    for s, rs in enumerate(res["names"]):
        for r, rr in enumerate(res["names"]):
            if res["A_rule"][s][r]:
                crit = (rr, rs) in path
                (x0, y0), (x1, y1) = xy[rr], xy[rs]
                ax.annotate("", (x1 - 0.36, y1), (x0 + 0.36, y0),
                            arrowprops=dict(arrowstyle="-|>", lw=2.2 if crit else 1.0,
                                            color="#dc2626" if crit else "#9ca3af"))  # fmt: skip
    for n, lv in zip(res["names"], res["levels"]):
        x, y = xy[n]
        on = lv < done_level
        ax.text(x, y, f"{SHORT[n]}\n{res['seconds'][n]:.0f} s" if on else SHORT[n], ha="center", va="center",
                fontsize=9, color="white" if on else "#374151",
                bbox=dict(boxstyle="round,pad=0.25", fc=RULE_COLOUR[n] if on else "white", ec=RULE_COLOUR[n],
                          lw=1.5))  # fmt: skip
    ax.set_xlim(-0.55, max(res["levels"]) + 0.55)
    ax.set_ylim(-1.6, 1.6)
    ax.set_xticks(range(max(res["levels"]) + 1))
    ax.set_yticks([])
    ax.set_xlabel("level $\\ell$ (dimensionless)", fontsize=10)
    ax.tick_params(labelsize=10)
    ax.set_title(f"training DAG: critical path {res['makespan']:.0f} s, serial {res['serial']:.0f} s", fontsize=11)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)


def _clip(text: str, n: int) -> str:
    return text if len(text) <= n else text[:n].rsplit(" ", 1)[0] + " …"


def _typed(text: str, frac: float) -> str:
    return text[: int(round(frac * len(text)))]


def _wrap(text: str, width: int = 26) -> str:
    words, lines, cur = text.split(), [], ""
    for w in words:
        if len(cur) + len(w) + (1 if cur else 0) > width:
            lines.append(cur)
            cur = w
        else:
            cur = f"{cur} {w}" if cur else w
    return "\n".join(lines + [cur] if cur else lines)


def animate(res: dict, path: Path, n_frames: int = 30, hold: int = 24) -> Path:
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(1, 4, width_ratios=[1.35, 1.05, 1.25, 1.05], wspace=0.4, left=0.04, right=0.985, top=0.93,
                          bottom=0.6)  # fmt: skip
    ax_dag, ax_loss = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    ax_f1, ax_cap = fig.add_subplot(gs[2]), fig.add_subplot(gs[3])
    bot = fig.add_gridspec(1, 3, width_ratios=[3.05, 0.9, 1.25], wspace=0.2, left=0.02, right=0.985, top=0.42,
                           bottom=0.06)  # fmt: skip
    ex_gs = bot[0].subgridspec(2, 2, hspace=0.6, wspace=0.06)
    ex_axes = []
    for i in range(4):  # image | gap | finding bars | (room for the finding names, right) | written reports
        sub = ex_gs[i // 2, i % 2].subgridspec(1, 5, width_ratios=[1.0, 0.1, 0.6, 1.08, 1.3], wspace=0.04)
        ex_axes.append((fig.add_subplot(sub[0]), fig.add_subplot(sub[2]), fig.add_subplot(sub[4])))
    g_axes = [fig.add_subplot(a) for a in bot[1].subgridspec(1, 2, wspace=0.3)]
    cap_gs = bot[2].subgridspec(2, 1, hspace=0.15)
    cap_axes = [fig.add_subplot(cap_gs[i]) for i in range(2)]
    S = len(res["seeds"])
    n_levels = max(res["levels"]) + 1
    loss = {k: v.numpy() for k, v in res["loss"].items() if v.numel()}
    lmax = max(float(np.nanmax(v)) for v in loss.values())
    lmin = min(float(np.nanmin(v)) for v in loss.values())
    jit = np.random.default_rng(0).uniform(-0.15, 0.15, S)

    # ---- static: router G, fixed vs learned (top-1 choice of the learned router's p)
    panels = (
        (res["G_fixed"], "fixed $G$\n(by data type)"),
        (res["G_learned"], f"learned router\nroute acc. {res['route_show']:.2f}"),
    )
    for i, (ax, (G, title)) in enumerate(zip(g_axes, panels)):
        im = ax.imshow(G, cmap="Blues", vmin=0, vmax=1, aspect="auto", interpolation="nearest")
        ax.axvline(T.K - 0.5, color="#dc2626", lw=1.5, ls="--")
        if i == 0:
            ax.set_ylabel("test image (dimensionless)", fontsize=10)
        else:
            ax.set_yticklabels([])
        ax.set_title(title, fontsize=11)
        ax.tick_params(labelsize=10)
        ax.set_xticks([0, 30])
    cb = fig.colorbar(im, ax=g_axes, fraction=0.05, pad=0.03)
    cb.set_label("$p_{t,\\mathrm{img}}$ (dimensionless)", fontsize=10)
    cb.ax.tick_params(labelsize=9)
    p0, p1 = g_axes[0].get_position(), g_axes[1].get_position()  # one x-label shared by both heatmaps
    fig.text(
        (p0.x0 + p1.x1) / 2, p0.y0 - 0.035, "input position $t$ (dimensionless)", ha="center", va="top", fontsize=10
    )

    def update(f: int) -> None:
        prog = min(1.0, f / max(1, n_frames - 1))
        show = f >= n_frames
        typed = min(1.0, (f - n_frames + 1) / max(1, hold * 0.7)) if show else 0.0
        _draw_dag(ax_dag, res, prog * n_levels if not show else n_levels + 1)

        ax_loss.clear()
        for rule, v in loss.items():
            k = max(1, int(round(prog * len(v))))
            steps = np.arange(1, len(v) + 1)
            ax_loss.plot(steps[:k], v[:k].mean(1), color=RULE_COLOUR[rule], lw=1.6, label=SHORT[rule])
        ax_loss.set_yscale("log")
        ax_loss.set_ylim(lmin * 0.7, lmax * 1.4)
        ax_loss.set_xlim(1, max(len(v) for v in loss.values()))
        ax_loss.set_xlabel("optimiser step (dimensionless)", fontsize=10)
        ax_loss.set_ylabel("loss $\\mathcal{L}$ (nats)", fontsize=10)
        ax_loss.set_title("per-stage loss (seed mean)", fontsize=11)
        ax_loss.legend(fontsize=9, loc="upper center", ncol=4, bbox_to_anchor=(0.5, -0.2), frameon=False,
                       handlelength=1.0, columnspacing=0.7)  # fmt: skip
        ax_loss.tick_params(labelsize=10)

        ax_f1.clear()
        for i, k in enumerate(STAGE):
            a = res["f1_micro"][k].numpy()
            ax_f1.bar(i, a.mean() if show else 0, color=STAGE[k][1], width=0.7)
            if show:
                ax_f1.scatter(i + jit, a, s=16, color="black", zorder=3)
        ax_f1.axhline(res["chance"], color="#9ca3af", lw=1.0, ls="--", label="random at prevalence")
        ax_f1.legend(fontsize=9, loc="upper right", framealpha=0.9)
        ax_f1.set_xticks(range(len(STAGE)), [v[0] for v in STAGE.values()], fontsize=9, rotation=35, ha="right",
                         rotation_mode="anchor")  # fmt: skip
        ax_f1.set_ylim(0, 0.6)
        ax_f1.set_xlim(-0.6, len(STAGE) - 0.4)
        ax_f1.set_ylabel("micro $F_1$ of six findings (dimensionless)", fontsize=10)
        ax_f1.set_title("NIH test: labels parsed from the report", fontsize=11)
        ax_f1.tick_params(axis="y", labelsize=10)

        ax_cap.clear()
        keys = list(CAP)
        for j, (metric, hatch) in enumerate((("rouge_l", ""), ("kw_recall", "//"))):
            for i, k in enumerate(keys):
                a = res["cap"][k][metric].numpy()
                ax_cap.bar(i + (j - 0.5) * 0.38, a.mean() if show else 0, width=0.36, color=CAP[k][1],
                           hatch=hatch, edgecolor="white")  # fmt: skip
                if show:
                    ax_cap.scatter(i + (j - 0.5) * 0.38 + jit * 0.5, a, s=10, color="black", zorder=3)
        ax_cap.legend(handles=[Patch(fc="#9ca3af", label="ROUGE-L"),
                               Patch(fc="#9ca3af", hatch="//", ec="white", label="keyword recall")],
                      fontsize=9, loc="upper left", framealpha=0.9)  # fmt: skip
        ax_cap.set_xticks(range(len(keys)), [CAP[k][0] for k in keys], fontsize=9, rotation=30, ha="right",
                          rotation_mode="anchor")  # fmt: skip
        ax_cap.set_ylim(0, 0.45)
        ax_cap.set_ylabel("score (dimensionless)", fontsize=10)
        ax_cap.set_title("PMC test captions (stage 2)", fontsize=11)
        ax_cap.tick_params(axis="y", labelsize=10)

        for (a_img, a_bar, a_txt), ex in zip(ex_axes, res["examples"]):
            a_img.clear()
            a_img.imshow(ex["img"], cmap="gray", vmin=0, vmax=255)
            a_img.set_xticks([])
            a_img.set_yticks([])
            a_bar.clear()
            a_bar.barh(np.arange(6), ex["q"] if show else np.zeros(6), color=ACCENT[5])
            a_bar.axvline(0.5, color="#9ca3af", lw=0.8, ls="--")
            a_bar.set_xlim(0, 1)
            a_bar.set_ylim(5.6, -0.6)
            a_bar.yaxis.tick_right()
            a_bar.set_yticks(range(6), T.FINDINGS, fontsize=9)
            a_bar.set_xticks([0, 0.5, 1], ["0", ".5", "1"], fontsize=9)
            a_bar.set_xlabel("$q_f$ (dimensionless)", fontsize=9)
            a_txt.clear()
            a_txt.axis("off")
            a_txt.set_xlim(0, 1)
            a_txt.set_ylim(0, 1)
            if show:
                ok = ex["pred"] == ex["truth"]
                a_txt.text(0.1, 1.0, "DAG report:", fontsize=9, va="top", color="#374151")
                a_txt.text(0.1, 0.84, _wrap(_typed(ex["dag"], typed), 16), fontsize=9, va="top",
                           family="monospace", color=ACCENT[2] if ok else ACCENT[1])  # fmt: skip
                a_txt.text(0.1, 0.36, "monolith:", fontsize=9, va="top", color="#374151")
                a_txt.text(0.1, 0.20, _wrap(_typed(ex["mono"], typed), 16), fontsize=9, va="top",
                           family="monospace", color="#6b7280")  # fmt: skip
            a_img.set_title("true:\n" + _wrap(ex["truth_short"], 16), fontsize=9, loc="left")

        for ax, ex in zip(cap_axes, res["cap_examples"]):
            ax.clear()
            ax.set_xticks([])
            ax.set_yticks([])
            ax.imshow(ex["img"], cmap="gray", vmin=0, vmax=255, extent=(0, 1, 0.55, 1.55))
            ax.set_xlim(0, 3.7)
            ax.set_ylim(-0.45, 1.58)
            ax.set_aspect("equal")
            for sp in ax.spines.values():
                sp.set_visible(False)
            ax.text(1.1, 1.52, "PMC text (start):\n" + _wrap(_clip(ex["truth"], 96), 32), fontsize=9, va="top",
                    color="#374151")  # fmt: skip
            if show:
                ax.text(1.1, 0.5, "model (start):\n" + _wrap(_clip(_typed(ex["gen"], typed), 96), 32),
                        fontsize=9, va="top",
                        family="monospace", color=ACCENT[3])  # fmt: skip
        fig.suptitle("Chest X-ray CNN + tiny report LM, trained apart and stitched by one DAG edge (a teaching toy, "
                     "not clinical report generation)", fontsize=13)  # fmt: skip

    # the first frame is the final state (an informative still), then the training replay, then the final hold
    return save_gif_shared_palette(fig, update, [n_frames + hold - 1] + list(range(n_frames + hold)), path)


def schedule_png(res: dict, path: Path) -> Path:
    """Measured rule times as a Gantt chart: one worker running the rules level by level (serial), and one worker
    per rule starting when its inputs are done (max-plus finish times); black edges = critical path."""
    names = res["names"]
    fig, ax = plt.subplots(figsize=(9, 4.6))
    t0, top = 0.0, len(names) + 0.6
    for i, n in enumerate(names):
        d, fin = res["seconds"][n], res["finish"][i]
        ax.barh(top, d, left=t0, height=0.7, color=RULE_COLOUR[n], edgecolor="white")
        crit = n in res["path"]
        ax.barh(len(names) - 1 - i, d, left=fin - d, height=0.6, color=RULE_COLOUR[n],
                edgecolor="#111827" if crit else "white", linewidth=2.0 if crit else 0.5)  # fmt: skip
        t0 += d
    labels = [SHORT[n] + (" (CP)" if n in res["path"] else "") for n in names][::-1] + ["serial"]
    ax.set_yticks(list(range(len(names))) + [top], labels, fontsize=10)
    ax.axhline(len(names) - 0.2, color="#9ca3af", lw=1, ls="--")
    ax.set_xlabel("wall time $t$ (s)", fontsize=10)
    ax.set_xlim(0, res["serial"] * 1.02)
    ax.set_ylim(-0.6, top + 0.6)
    ax.tick_params(labelsize=10)
    ax.set_title(f"measured schedule: serial {res['serial']:.0f} s; one worker per rule {res['makespan']:.0f} s "
                 "(critical path, CP)", fontsize=11)  # fmt: skip
    fig.tight_layout()
    return save_png(fig, path)


def write_tables(res: dict, out_dir: Path) -> None:
    with open(out_dir / "stages.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["stage", "seed", "micro_f1", "macro_f1", "wellformed_template", "exact_template"])
        for k in STAGE:
            for si, s in enumerate(res["seeds"]):
                r = [res[m][k][si] if k in res[m] else float("nan") for m in ("f1_micro", "f1_macro", "wellformed",
                                                                                "exact")]  # fmt: skip
                w.writerow([k, s] + [f"{float(v):.4f}" for v in r])
        for si, s in enumerate(res["seeds"]):
            w.writerow(["cnn_mean_auroc", s, f"{float(res['auc'][si].mean()):.4f}", "", "", ""])
            w.writerow(["router_routing_accuracy", s, f"{float(res['route_acc'][si]):.4f}", "", "", ""])
        w.writerow(["chance_micro_f1_random_at_prevalence", "", f"{res['chance']:.4f}", "", "", ""])
    with open(out_dir / "captions.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["model", "seed", "bleu1", "rouge_l", "keyword_recall", "keyword_precision",
                    "heldout_nats_per_letter"])  # fmt: skip
        for k, c in res["cap"].items():
            for si, s in enumerate(res["seeds"]):
                w.writerow([k, s] + [f"{float(c[m][si]):.4f}" for m in ("bleu1", "rouge_l", "kw_recall",
                                                                        "kw_precision", "nats")])  # fmt: skip
    with open(out_dir / "caption_examples.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["pmc_test_index", "true_caption_cleaned", "stage1_model", "caption_model", "shuffled_images"])
        for i, row in enumerate(res["cap_rows"]):
            w.writerow([i, *row])
    with open(out_dir / "report_examples.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["nih_test_index", "true_findings", "cnn_probabilities", "dag_report", "monolith_report"])
        for i, row in enumerate(res["rep_rows"]):
            w.writerow([i, *row])
    with open(out_dir / "timings.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["rule", "level", "seconds", "finish_one_worker_per_rule_s", "on_critical_path"])
        for i, n in enumerate(res["names"]):
            w.writerow([n, res["levels"][i], f"{res['seconds'][n]:.2f}", f"{res['finish'][i]:.2f}",
                        int(n in res["path"])])  # fmt: skip


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'seeds': 2,
    'steps_cnn': 250,
    'steps_lm': 200,
    'steps_con': 120,
    'steps_ft': 60,
    'steps_router': 40,
    'steps_cap': 150,
    'lr_ft': 0.0003,
    'lr_cap': 0.01
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.ai.cxr_report_dag")]:
    del sys.modules[name]
main = importlib.import_module("problems.ai.cxr_report_dag.main")
problem_dir = Path("problems/ai/cxr_report_dag")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
